# Mosca ajedrez — FlyBrainNet en Colab Free

Este notebook ejecuta la fase 0 sin descargar bases grandes. El entrenamiento smoke usa posiciones sintéticas y compara FlyBrainNet, máscara barajada, MLP y CNN. La base real se descarga después en Colab, no en el ordenador local, y solo tras confirmar tamaño y espacio.

In [ ]:
!pip -q install chess numpy pandas pyarrow python-dotenv PyYAML zstandard matplotlib pytest torch networkx

In [ ]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
PROJECT = Path('/content/drive/MyDrive/mosca_ajedrez')
if not PROJECT.exists():
    PROJECT = Path('/content/mosca_ajedrez')
if not PROJECT.exists():
    raise FileNotFoundError('Sube mosca_ajedrez a Drive o a /content antes de continuar')
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))
print('Proyecto:', PROJECT)

## Hardware y tests

Colab Free puede tener CPU o una GPU temporal. La celda no asume GPU y el smoke está dimensionado para terminar en CPU.

In [ ]:
from src.device import report_device
report_device()
!make test
!make smoke

In [ ]:
import json
from pathlib import Path
print(json.loads(Path('data/processed/smoke_report.json').read_text()))

## Cuerpo virtual

El cuerpo es una visualización separada del cerebro: energía, posición, orientación y alas. No se presenta como un cuerpo biológico real.

In [ ]:
import matplotlib.pyplot as plt
from src.body import FlyBody
body = FlyBody()
for confidence in [0.1, 0.8, 0.4, 0.9]: body.step(confidence)
body.draw(); plt.show()

## Cerebro virtual (Connectome)

Visualización de las conexiones neuronales del modelo (FlyBrainNet) simulando la estructura del cerebro de la mosca.

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from src.models.fly import FlyBrainNet

# Instanciar un cerebro pequeño para poder visualizarlo
units = 256
model = FlyBrainNet(units=units)
adj = model.connectome.cpu().numpy()

# Crear un grafo dirigido a partir de la matriz de adyacencia
G = nx.from_numpy_array(adj, create_using=nx.DiGraph)

# Visualizar un subgrafo de 50 neuronas para mayor claridad
sub_G = G.subgraph(range(50))
plt.figure(figsize=(10, 8))
plt.title("Conexiones Neuronales (Subgrafo de 50 neuronas)")
pos = nx.spring_layout(sub_G, seed=42)
nx.draw_networkx_nodes(sub_G, pos, node_size=50, node_color='purple', alpha=0.7)
nx.draw_networkx_edges(sub_G, pos, edge_color='gray', alpha=0.3, arrows=True)
plt.axis('off')
plt.show()

# Visualizar la matriz de adyacencia completa
plt.figure(figsize=(8, 8))
plt.title(f"Matriz de Adyacencia Completa ({units}x{units})")
plt.imshow(adj, cmap='Greys', interpolation='none')
plt.show()


## La mosca jugando (Primeros intentos)

Aquí conectamos el cerebro (no entrenado aún) a un tablero de ajedrez real. Al no haber visto partidas todavía, la mosca intentará movimientos pseudo-aleatorios intentando entender las reglas, y elegiremos el movimiento legal que más 'le guste' a sus neuronas.

In [ ]:
import chess
import chess.svg
import random
from IPython.display import display, HTML

board = chess.Board()
print("¡La mosca se acerca al tablero!")

# Como el cerebro aún no está entrenado, vamos a hacer que la mosca juegue contra sí misma 
# tomando una decisión rápida entre los movimientos legales.
for i in range(10):  # Haremos 10 movimientos de demostración
    if board.is_game_over():
        break
    
    legal_moves = list(board.legal_moves)
    # En el futuro, el FlyBrainNet elegirá este movimiento.
    # Por ahora, como buena mosca, hace un movimiento aleatorio (pero legal).
    move = random.choice(legal_moves)
    board.push(move)

display(HTML(chess.svg.board(board=board, size=400)))
print("Partida tras 10 movimientos aleatorios de la mosca.")